# LoRA fine-tune on Google Colab (T4, free tier)

This notebook fine-tunes a small open-weight model (default: `Llama-3.2-3B-Instruct` in 4-bit) on the Alpaca JSONL exported from the studio, then converts the result to **GGUF q4\_k\_m** so you can run it locally in **Ollama**.

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`. The free T4 has ~16 GB VRAM, which is enough for a 3B model + LoRA at seq-length 2048.

Expected wall-time on a free T4 with ~1k Alpaca rows, 2 epochs, LoRA r=16:
- install step: ~2-3 min
- model download: ~1 min (first run only)
- training: ~10-20 min
- GGUF export: ~5-10 min (llama.cpp build dominates)

The notebook calls `studio/scripts/train_lora.py` and `studio/scripts/export_gguf.py` from the repo; cell flags are the knobs you tune per run.

In [ ]:
# Cell 2 - confirm a GPU is attached. You want to see 'Tesla T4' (or better).
!nvidia-smi

In [ ]:
# Cell 3 - install dependencies.
# Unsloth ships its own patched versions of transformers/peft/trl, so on
# Colab we just let pip resolve the latest stable wheels.
!pip install unsloth datasets accelerate peft bitsandbytes transformers trl -q

In [ ]:
# Cell 4 - clone the studio repo. Already cloned? Second run is a no-op.
%cd /content
import os
if not os.path.isdir('/content/jailbreak-pipeline-studio'):
    !git clone https://github.com/rkstlohchab/jailbreak-pipeline-studio.git
%cd /content/jailbreak-pipeline-studio
!pwd && ls studio/scripts | head

In [ ]:
# Cell 5 - peek at the dataset you are about to train on.
# Pick whichever file actually exists: the studio export, or the bundled
# 3-row sample when you just want to smoke-test the pipeline.
import json, os

candidates = [
    'studio/data/exports/alpaca.jsonl',
    'studio/scripts/_sample_alpaca.jsonl',
]
dataset_path = next((p for p in candidates if os.path.exists(p)), None)
assert dataset_path is not None, (
    'No alpaca.jsonl found - export one from the studio first, or upload '
    'it to /content/jailbreak-pipeline-studio/studio/data/exports/.'
)

print(f'using dataset: {dataset_path}')
with open(dataset_path, 'r', encoding='utf-8') as fh:
    rows = [json.loads(line) for line in fh if line.strip()]
print(f'rows: {len(rows)}')
for i, row in enumerate(rows[:2]):
    print(f'\n--- row {i} ---')
    print(json.dumps(row, indent=2)[:600])

In [ ]:
# Cell 6 - train the LoRA adapter.
# Knobs to tune on a free T4:
#   --model           smaller base = faster, less VRAM
#   --epochs          more epochs = more overfit on tiny datasets
#   --batch-size 2 --grad-accum 4   effective batch 8, fits in 16 GB
#   --max-seq-length  drop to 1024 if you OOM on long samples
#   --merge           also save merged 16-bit weights for the next cell
!python studio/scripts/train_lora.py \
    --model unsloth/Llama-3.2-3B-Instruct-bnb-4bit \
    --dataset {dataset_path} \
    --output studio/data/finetunes/custom-llama3.2-3b \
    --epochs 2 --lora-r 16 --lora-alpha 32 \
    --batch-size 2 --grad-accum 4 \
    --max-seq-length 2048 \
    --merge

In [ ]:
# Cell 7 - convert the LoRA-merged model to GGUF q4_k_m for Ollama.
# This needs llama.cpp (convert_hf_to_gguf.py + llama-quantize binary).
# Build is cached across re-runs of the same Colab VM.
import os

if not os.path.isdir('/content/llama.cpp'):
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
    !pip install -r /content/llama.cpp/requirements.txt -q

if not os.path.exists('/content/llama.cpp/build/bin/llama-quantize'):
    !cd /content/llama.cpp && cmake -B build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /tmp/cmake.log 2>&1
    !cd /content/llama.cpp && cmake --build build --config Release -j --target llama-quantize > /tmp/build.log 2>&1
    !ls /content/llama.cpp/build/bin/llama-quantize

!python studio/scripts/export_gguf.py \
    --adapter studio/data/finetunes/custom-llama3.2-3b \
    --base    unsloth/Llama-3.2-3B-Instruct \
    --output  studio/data/finetunes/custom-llama3.2-3b.gguf \
    --quant   q4_k_m \
    --llama-cpp /content/llama.cpp

In [ ]:
# Cell 8 - download the GGUF off Colab.
# Option A: direct browser download (best for files <~2 GB).
# Option B: copy to Google Drive (survives VM restarts, handles larger files).
import os

gguf_path = 'studio/data/finetunes/custom-llama3.2-3b.gguf'
assert os.path.exists(gguf_path), f'missing: {gguf_path}'
size_mib = os.path.getsize(gguf_path) / (1 << 20)
print(f'GGUF size: {size_mib:.1f} MiB')

USE_DRIVE = size_mib > 1800  # flip to True to force Drive
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    dest = '/content/drive/MyDrive/finetunes/'
    os.makedirs(dest, exist_ok=True)
    !cp {gguf_path} {dest}
    print(f'copied to Drive: {dest}')
else:
    from google.colab import files
    files.download(gguf_path)

## Next steps - run it locally in Ollama

On your own machine, once the GGUF has finished downloading:

```bash
# 1. Put the .gguf anywhere on disk.
mkdir -p ~/models/custom-llama3.2-3b && cd ~/models/custom-llama3.2-3b
mv ~/Downloads/custom-llama3.2-3b.gguf .

# 2. Create a Modelfile next to it (the export_gguf.py run above printed
#    the exact template you can paste here).
cat > Modelfile <<'EOF'
FROM ./custom-llama3.2-3b.gguf
PARAMETER temperature 0.7
PARAMETER top_p       0.9
PARAMETER num_ctx     4096
EOF

# 3. Register it with Ollama and run a quick test.
ollama create custom-llama3.2-3b -f Modelfile
ollama run    custom-llama3.2-3b 'Summarize the plot of Macbeth in two sentences.'
```

**Iterating:**
- Change `--epochs`, `--lora-r`, or the base model in **Cell 6** and rerun 6-8. Adapter dir is overwritten each time.
- Low VRAM? Drop `--max-seq-length` to 1024, or switch to `unsloth/Llama-3.2-1B-Instruct-bnb-4bit`.
- Want a different quant? Change `--quant q4_k_m` in **Cell 7** to `q5_k_m` (higher quality, ~25% larger) or `q8_0` (near-lossless, ~2x larger).
- See `studio/scripts/README_TRAINING.md` for the full FAQ and base-model menu.